In [1]:
import sys
!{sys.executable} -m pip install openpyxl


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 1. Import Library
import pandas as pd
import os
import glob

print("Library dimuat")

Library dimuat


In [3]:
# 2. Definisi Path Folder
folder_path = '../outputs/evaluation/annotators_raw'
output_path = '../outputs/evaluation'

# Cek apakah folder ada
if not os.path.exists(folder_path):
    print(f"ERROR: Folder '{folder_path}' tidak ditemukan. Cek kembali path-nya.")
else:
    # Ambil semua file .xlsx di folder tersebut
    file_list = glob.glob(os.path.join(folder_path, "*.xlsx"))
    print(f"Ditemukan {len(file_list)} file Excel untuk digabungkan.")
    
    # Inisialisasi list untuk menyimpan dataframe
    dfs = []
    
    # Loop untuk membaca setiap file
    for i, file in enumerate(file_list):
        # Ambil nama file untuk identifier
        annotator_name = os.path.basename(file).replace('.xlsx', '')
        
        print(f"Membaca file: {file} ...")
        
        # Baca Excel
        try:
            df_temp = pd.read_excel(file, sheet_name='ground_truth.csv')
            
            # Ambil hanya kolom yang dibutuhkan
            df_clean = df_temp[['no', 'ground_truth_score', 'ground_truth_category']].copy()
            
            # Ubah nama kolom score biar tidak bentrok
            df_clean.rename(columns={'ground_truth_score': f'score_{annotator_name}'}, inplace=True)
            df_clean.rename(columns={'ground_truth_category': f'category_{annotator_name}'}, inplace=True)
            
            # Tipe data score adalah angka
            df_clean[f'score_{annotator_name}'] = pd.to_numeric(df_clean[f'score_{annotator_name}'], errors='coerce')
            
            dfs.append(df_clean)
            
        except Exception as e:
            print(f"Gagal membaca {file}. Error: {e}")
            print("Pastikan nama sheet 'ground_truth.csv' dan kolom 'no'/'ground_truth_score' ada di file Excel.")

Ditemukan 5 file Excel untuk digabungkan.
Membaca file: ../outputs/evaluation/annotators_raw\annotator 1.xlsx ...
Membaca file: ../outputs/evaluation/annotators_raw\annotator 2.xlsx ...
Membaca file: ../outputs/evaluation/annotators_raw\annotator 3.xlsx ...
Membaca file: ../outputs/evaluation/annotators_raw\annotator 4.xlsx ...
Membaca file: ../outputs/evaluation/annotators_raw\annotator 5.xlsx ...


In [4]:
# 3. Menggabungkan Data (Merge)
if len(dfs) > 0:
    df_combined = dfs[0]
    
    for df in dfs[1:]:
        df_combined = pd.merge(df_combined, df, on='no', how='inner')
        
    print("\nData berhasil digabungkan!")
    print(f"Total baris: {len(df_combined)}")
    print(f"Kolom: {df_combined.columns.tolist()}")
    
    print("\nPreview Data Gabungan:")
    print(df_combined.head())
    
    output_file = os.path.join(output_path, 'combined_annotations.csv')
    df_combined.to_csv(output_file, index=False)
    print(f"\nFile gabungan tersimpan di: {output_file}")
else:
    print("Tidak ada data yang berhasil dibaca.")


Data berhasil digabungkan!
Total baris: 450
Kolom: ['no', 'score_annotator 1', 'category_annotator 1', 'score_annotator 2', 'category_annotator 2', 'score_annotator 3', 'category_annotator 3', 'score_annotator 4', 'category_annotator 4', 'score_annotator 5', 'category_annotator 5']

Preview Data Gabungan:
     no  score_annotator 1 category_annotator 1  score_annotator 2  \
0  6885                  0              Neutral                 -1   
1  7320                  0              Neutral                  0   
2   360                  0              Neutral                  0   
3  1514                  0              Neutral                  0   
4  4881                  0              Neutral                  0   

  category_annotator 2  score_annotator 3 category_annotator 3  \
0    Slightly Negative                 -1    Slightly Negative   
1              Neutral                  1    Slightly Positive   
2              Neutral                  0              Neutral   
3      